### 自己实现

In [4]:
import os
import jieba
from tqdm import tqdm
from collections import Counter
import math

def word_cut(text):
    cuts_generator = jieba.cut(text, cut_all=False)
    return [word for word in cuts_generator if len(word) > 1]

def del_word(words, del_words):
    return [word for word in words if word not in del_words]

def prepare(text):
    with open('./data/stopwords.dat', 'r') as fr:
        stop_words = [x.strip() for x in fr.readlines()]

    words = word_cut(text.strip())
    words = del_word(words, stop_words)
    return words

def get_dataset():
    words_list = []

    # 读取训练数据
    files = os.listdir('./data/THUCNews/教育')
    for i, file in tqdm(enumerate(files)):
        if i == 1000:
            break
        sents = []
        with open('./data/THUCNews/教育/' + file, 'r') as fr:
            sentences = fr.readlines()

        for sent in sentences:
            sents += prepare(sent)
            
        words_list.append(sents)
        
    return words_list

def doc_count(words_list):
    count_list = []
    for words in words_list:
        count = Counter(words)
        count_filter = {}
        for word, cnt in count.items():
            if cnt > 1:
                count_filter[word] = cnt
        count_list.append(count_filter)
    return count_list

def tf(word, count):  # count 是单个文档
    return count[word] / sum(count.values())

def idf(word, count_list):
    n_contain = sum([1 for count in count_list if word in count])
    return math.log(len(count_list) / (1 + n_contain))

def tf_idf(word, count, count_list):
    return tf(word, count) * idf(word, count_list)

def tfidf_compute():
    for i, count in enumerate(count_list[:3]):
        print(f'第 {i} 个文档 TF-IDF 统计信息')
        scores = {word: tf_idf(word, count, count_list) for word in count}
        sorted_word = sorted(scores.items(), key = lambda x: x[1], reverse=True)
        for word, score in sorted_word[:3]:
            print(f'\t 词语: {word}, TF-IDF: {round(score, 5)}')


words_list = get_dataset()
count_list = doc_count(words_list)
tfidf_compute()

0it [00:00, ?it/s]Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/rj/kh58t7hj5s99k8krgn4brl1c0000gn/T/jieba.cache
Loading model cost 0.319 seconds.
Prefix dict has been built successfully.
1000it [00:10, 92.97it/s]

第 0 个文档 TF-IDF 统计信息
	 词语: 项目, TF-IDF: 0.18592
	 词语: 子女, TF-IDF: 0.15451
	 词语: 文凭, TF-IDF: 0.14455
第 1 个文档 TF-IDF 统计信息
	 词语: 留学, TF-IDF: 0.49476
	 词语: 职业规划, TF-IDF: 0.16717
	 词语: 国外, TF-IDF: 0.13894
第 2 个文档 TF-IDF 统计信息
	 词语: 语法, TF-IDF: 0.40518
	 词语: 单词, TF-IDF: 0.15664
	 词语: 词汇, TF-IDF: 0.13761


### gensim 算法包

In [5]:
from gensim import corpora, models, matutils

def get_dataset(words_list):
    # 赋给语料库中每个词(不重复的词)一个整数id
    dic = corpora.Dictionary(words_list)
    # print(dic.token2id)
    # 元组中第一个元素是词语在词典中对应的id，第二个元素是词语在文档中出现的次数
    new_corpus = [dic.doc2bow(words) for words in words_list]
    # print(new_corpus)
    dic.save('./data/tfidf_dict.dict')
    return new_corpus

def build_tfidf(data):
    tfidf = models.TfidfModel(data)
    tfidf.save('./model/tfidf.model')
        
def test(test_data):
    print(test_data)
    # 载入模型
    dic = corpora.Dictionary.load('./data/tfidf_dict.dict')
    corpus = [dic.doc2bow(doc) for doc in test_data]
    model = models.TfidfModel.load('./model/tfidf.model')
    vocab_size = len(dic.token2id)

    for doc in corpus:
        print(model[doc])

# words_list = get_dataset()
data = get_dataset(words_list)
build_tfidf(data)
test([prepare('中国现在出国留学的人在增加')])

[['中国', '出国', '留学', '增加']]
[(12, np.float64(0.3440503613866316)), (36, np.float64(0.6508484230857554)), (66, np.float64(0.3916132505968811)), (144, np.float64(0.551964438123923))]


### sklearn 算法包

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# documents = [' '.join(words) for words in words_list]
documents = ['中国 北京 中国',
             '中国 中国 上海',
             '中国 悉尼',
             '东京 日本 中国']

# 创建 TfidfVectorizer 对象
vectorizer = TfidfVectorizer()

# 将文本转换为TF-IDF表示
tfidf_matrix = vectorizer.fit_transform(documents)

print('train ---')
# 输出特征名字（单词）
print(len(vectorizer.get_feature_names_out()), vectorizer.get_feature_names_out())
# print(vectorizer.vocabulary_)

# 输出TF-IDF矩阵
print(tfidf_matrix.toarray())

print('test ---')
for item in [['中国 中国 中国 悉尼 东京']]:
    print(vectorizer.transform(item).toarray())

train ---
6 ['上海' '东京' '中国' '北京' '悉尼' '日本']
[[0.         0.         0.722056   0.69183461 0.         0.        ]
 [0.69183461 0.         0.722056   0.         0.         0.        ]
 [0.         0.         0.46263733 0.         0.88654763 0.        ]
 [0.         0.66338461 0.34618161 0.         0.         0.66338461]]
test ---
[[0.        0.4739993 0.7420575 0.        0.4739993 0.       ]]
